# Vision-Based Lane Inference for Indian Roads — Model Training (Phase 4)

**BITS Pilani · BSc Computer Science · Study Project**
Shreyas Bhat K (2023EBCS460) · Harshwardhan Mukund Mohadikar (2023EBCS353)
Supervisor: Dr. Ashok Yemineni

---

This notebook trains the drivable-area / semantic segmentation network used by the
Phase 4 lane-inference pipeline, on **IDD-Lite** (Indian Driving Dataset, level-1
label hierarchy — 1,403 train / 204 val frames at 320×227, 7 classes).

**Before running:** `Runtime → Change runtime type → T4 GPU`.

Run the cells top to bottom. Total runtime on a T4 is roughly 15–25 minutes.

## 1. Verify the GPU runtime

In [ ]:
import subprocess, torch, platform
print(subprocess.run(['nvidia-smi','--query-gpu=name,memory.total,driver_version',
                      '--format=csv,noheader'], capture_output=True, text=True).stdout.strip()
      or 'No NVIDIA GPU visible')
print('python  :', platform.python_version())
print('torch   :', torch.__version__)
print('cuda    :', torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')
assert torch.cuda.is_available(), \
    'No GPU. Set Runtime > Change runtime type > T4 GPU, then re-run.'

## 2. Load the project bundle

`odp_bundle.zip` contains both the source tree (`lane_inference/`) and the
dataset (`datasets/idd_lite/`), about 32 MB.

The cell below first looks for the bundle on Google Drive. If it is not there it
falls back to a manual upload. Putting it on Drive is recommended — it survives
runtime restarts and lets you resume without re-uploading.

**Drive path expected:** `MyDrive/ODP/odp_bundle.zip`

In [ ]:
import os, shutil, zipfile, pathlib

BUNDLE = None
DRIVE_DIR = None

# --- Try Google Drive first -------------------------------------------------
try:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_DIR = pathlib.Path('/content/drive/MyDrive/ODP')
    DRIVE_DIR.mkdir(parents=True, exist_ok=True)
    candidate = DRIVE_DIR / 'odp_bundle.zip'
    if candidate.is_file():
        BUNDLE = candidate
        print('Found bundle on Drive:', BUNDLE)
    else:
        print('No bundle at', candidate, '- will ask for an upload.')
except Exception as e:
    print('Drive unavailable (%s) - will ask for an upload.' % e)

# --- Fall back to manual upload --------------------------------------------
if BUNDLE is None:
    from google.colab import files
    print('Select odp_bundle.zip ...')
    up = files.upload()
    name = next(iter(up))
    BUNDLE = pathlib.Path('/content') / name
    if DRIVE_DIR is not None:                 # cache it for next time
        shutil.copy(BUNDLE, DRIVE_DIR / 'odp_bundle.zip')
        print('Cached to Drive for future runs.')

# --- Unpack -----------------------------------------------------------------
ROOT = pathlib.Path('/content/odp')
if ROOT.exists():
    shutil.rmtree(ROOT)
ROOT.mkdir(parents=True)
with zipfile.ZipFile(BUNDLE) as z:
    z.extractall(ROOT)

print('\nUnpacked to', ROOT)
for p in sorted(ROOT.iterdir()):
    print(' ', p.name)

## 3. Dependencies

Colab already ships torch, torchvision, OpenCV and NumPy, so this is usually a no-op.

In [ ]:
import importlib, subprocess, sys

for mod, pkg in [('cv2', 'opencv-python-headless'), ('torchvision', 'torchvision'),
                 ('scipy', 'scipy'), ('certifi', 'certifi')]:
    try:
        importlib.import_module(mod)
        print(f'{mod:<14} ok')
    except ImportError:
        print(f'{mod:<14} installing {pkg} ...')
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', pkg], check=True)

import cv2, numpy, torchvision
print('\nopencv     :', cv2.__version__)
print('numpy      :', numpy.__version__)
print('torchvision:', torchvision.__version__)

## 4. Sanity checks — dataset, labels, model

In [ ]:
import os, sys
os.chdir('/content/odp/lane_inference')
sys.path.insert(0, '/content/odp/lane_inference')
os.environ['PYTHONPATH'] = '/content/odp/lane_inference'

import numpy as np, torch
import config as cfg
from data.idd import IDDLite, class_frequencies

# Colab's copy of the data lives beside the code, not at the default path.
cfg.DATA_ROOT = __import__('pathlib').Path('/content/odp/datasets/idd_lite')

train_ds = IDDLite(cfg.DATA_ROOT, 'train')
val_ds   = IDDLite(cfg.DATA_ROOT, 'val', augment=False)
print(f'train frames : {len(train_ds)}')
print(f'val frames   : {len(val_ds)}')

freq = class_frequencies(cfg.DATA_ROOT, 'train')
print('\nclass pixel distribution (train split)')
for name, f in zip(cfg.CLASS_NAMES, freq):
    print(f'  {name:<18} {100*f/freq.sum():6.2f} %')

In [ ]:
# Visual confirmation that image/label pairs are aligned and class 0 is the road.
import matplotlib.pyplot as plt
from utils.image_utils import colorize_labels

fig, axes = plt.subplots(3, 3, figsize=(15, 9))
for col, idx in enumerate([5, 400, 900]):
    img, lbl = train_ds.raw(idx)
    colored = colorize_labels(lbl, cfg.CLASS_COLORS_BGR)
    blend = (0.55 * img + 0.45 * colored).astype('uint8')
    for row, (pic, title) in enumerate([(img, 'image'), (colored, 'ground truth'),
                                        (blend, 'overlay')]):
        axes[row][col].imshow(pic[:, :, ::-1])
        axes[row][col].set_title(f'{title} #{idx}', fontsize=9)
        axes[row][col].axis('off')
plt.tight_layout(); plt.show()

print('legend:', ', '.join(f'{i}={n}' for i, n in enumerate(cfg.CLASS_NAMES)))

In [ ]:
# Model size and single-frame latency on this GPU.
import time
from models.segnet import build_model, count_parameters

dev = 'cuda'
model = build_model(pretrained=True).to(dev).eval()
print(f'trainable parameters : {count_parameters(model)/1e6:.2f} M')

x = torch.randn(1, 3, 224, 320, device=dev)
with torch.no_grad():
    for _ in range(10):
        model(x)
    torch.cuda.synchronize()
    t = time.time()
    for _ in range(100):
        model(x)
    torch.cuda.synchronize()
dt = (time.time() - t) / 100
print(f'inference latency    : {dt*1000:.2f} ms  ({1/dt:.0f} FPS)')

## 5. Train

Configuration: MobileNetV3-Large encoder (ImageNet-pretrained) + LR-ASPP context
+ FPN decoder, AdamW with cosine decay and linear warm-up, weighted
cross-entropy + Dice loss, and an augmentation suite targeting the failure modes
documented in Phase 3 (over-exposure, cast shadows, motion blur, low resolution).

The best-mIoU checkpoint is written to `outputs/checkpoints/seg_mnv3/best.pt`.

In [ ]:
# -u = unbuffered, so epoch lines appear live instead of all at the end.
!cd /content/odp/lane_inference && PYTHONPATH=. \
    python -u -m models.train \
        --epochs 150 \
        --batch-size 32 \
        --lr 6e-4 \
        --num-workers 2 \
        --run-name seg_mnv3 \
        --data-root /content/odp/datasets/idd_lite


In [ ]:
# If the training cell ever looks frozen, run this afterwards (or from the
# Colab Terminal) - history.json is rewritten after every single epoch.
import json, pathlib
p = pathlib.Path('/content/odp/outputs/checkpoints/seg_mnv3/history.json')
if p.is_file():
    h = json.loads(p.read_text())
    print(f'{len(h)} epochs done; last: mIoU {h[-1]["val_miou"]:.4f}, '
          f'drivable {h[-1]["val_drivable_iou"]:.4f}, {h[-1]["seconds"]:.1f}s/epoch')
    print(f'best so far: mIoU {max(x["val_miou"] for x in h):.4f}')
else:
    print('no history yet')


## 6. Training curves

In [ ]:
import json, matplotlib.pyplot as plt

hist = json.load(open('/content/odp/outputs/checkpoints/seg_mnv3/history.json'))
ep   = [h['epoch'] for h in hist]

fig, ax = plt.subplots(1, 3, figsize=(16, 4))
ax[0].plot(ep, [h['train_loss'] for h in hist]);      ax[0].set_title('training loss')
ax[1].plot(ep, [h['val_miou'] for h in hist], label='mIoU')
ax[1].plot(ep, [h['val_drivable_iou'] for h in hist], label='drivable IoU')
ax[1].legend(); ax[1].set_title('validation IoU')
per = list(zip(*[h['per_class_iou'] for h in hist]))
for i, series in enumerate(per):
    ax[2].plot(ep, series, label=cfg.CLASS_NAMES[i], linewidth=1)
ax[2].legend(fontsize=7); ax[2].set_title('per-class validation IoU')
for a in ax:
    a.set_xlabel('epoch'); a.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('/content/odp/outputs/training_curves.png', dpi=150, bbox_inches='tight')
plt.show()

best = max(hist, key=lambda h: h['val_miou'])
print(f"best epoch {best['epoch']}  mIoU {best['val_miou']:.4f}  "
      f"drivable IoU {best['val_drivable_iou']:.4f}")
print(f"total training time: {sum(h['seconds'] for h in hist)/60:.1f} min")

## 7. Final validation metrics

In [ ]:
import importlib, torch
from torch.utils.data import DataLoader
import eval.metrics as M
importlib.reload(M)
from models.segnet import build_model

ck = torch.load('/content/odp/outputs/checkpoints/seg_mnv3/best.pt', map_location='cuda')
model = build_model(ck['arch'], pretrained=False).to('cuda')
model.load_state_dict(ck['model']); model.eval()
print(f"loaded checkpoint from epoch {ck['epoch']} (mIoU {ck['miou']:.4f})\n")

cm  = M.ConfusionMatrix()
bnd = M.BoundaryAccumulator(tolerance=3)
loader = DataLoader(val_ds, batch_size=32, shuffle=False, num_workers=2)

with torch.no_grad():
    for x, y in loader:
        pred = model(x.to('cuda')).argmax(1).cpu().numpy()
        gt   = y.numpy()
        cm.update(pred, gt)
        for p, g in zip(pred, gt):
            bnd.update(p == cfg.DRIVABLE_ID, g == cfg.DRIVABLE_ID)

summary = cm.summary()
print(M.format_table(summary, 'IDD-Lite validation (204 frames)'))
print()
for k, v in bnd.score().items():
    print(f'drivable {k:<20}: {v:.4f}')

## 8. Qualitative results

In [ ]:
import numpy as np, matplotlib.pyplot as plt
from data.idd import preprocess_bgr
from utils.image_utils import colorize_labels

picks = [3, 27, 64, 101, 150, 190]
fig, axes = plt.subplots(len(picks), 3, figsize=(13, 2.6*len(picks)))
with torch.no_grad():
    for r, idx in enumerate(picks):
        img, gt = val_ds.raw(idx)
        pred = model(preprocess_bgr(img, (224, 320)).to('cuda')).argmax(1)[0].cpu().numpy()
        for c, (pic, title) in enumerate([
                (img, 'input'),
                (colorize_labels(gt, cfg.CLASS_COLORS_BGR), 'ground truth'),
                (colorize_labels(pred.astype('uint8'), cfg.CLASS_COLORS_BGR), 'prediction')]):
            axes[r][c].imshow(pic[:, :, ::-1]); axes[r][c].axis('off')
            if r == 0:
                axes[r][c].set_title(title)
plt.tight_layout()
plt.savefig('/content/odp/outputs/qualitative_val.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. Save results

Writes the checkpoint, training history and figures back to Drive, and also
offers a direct download. Bring `seg_mnv3.zip` back into the local project at
`outputs/checkpoints/` so the rest of the pipeline (inverse-perspective mapping,
lane inference, video demo) can use it.

In [ ]:
import shutil, pathlib, zipfile

out = pathlib.Path('/content/odp/outputs')
pack = pathlib.Path('/content/seg_mnv3.zip')
with zipfile.ZipFile(pack, 'w', zipfile.ZIP_DEFLATED) as z:
    for p in (out / 'checkpoints' / 'seg_mnv3').rglob('*'):
        if p.is_file():
            z.write(p, p.relative_to(out))
    for fig in ['training_curves.png', 'qualitative_val.png']:
        if (out / fig).is_file():
            z.write(out / fig, fig)

print('packaged:', pack, f'{pack.stat().st_size/1e6:.1f} MB')
with zipfile.ZipFile(pack) as z:
    for n in z.namelist():
        print('  ', n)

try:
    dest = pathlib.Path('/content/drive/MyDrive/ODP/seg_mnv3.zip')
    dest.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy(pack, dest)
    print('\nsaved to Drive:', dest)
except Exception as e:
    print('\nDrive copy skipped:', e)

from google.colab import files
files.download(str(pack))